# ML-эксперименты E1–E6 — дипломная работа

Пошаговый прогон шести экспериментов поверх обученной v3-модели:

| # | Эксперимент | Требует весов | Требует GPU | Время |
|---|---|---|---|---|
| E3 | Score-card per-class | нет | нет | ~5 мин |
| E1 | Grad-CAM | да | желательно | ~5 мин CPU / ~1 мин GPU |
| E2 | Embeddings + t-SNE/UMAP | да | желательно | ~5 мин |
| E4 | Калибровка вероятностей | да | нет | ~3 мин |
| E5 | CLIP linear-probe + zero-shot | нет (CLIP качается) | желательно | ~10 мин GPU |
| E6 | Multi-label fine-tune | warm-start от v3 | **обязательно** | ~30 мин T4 |

**Включить GPU:** Runtime → Change runtime type → **T4 GPU**.

## Что должно лежать в Drive (на твоей стороне)

```
MyDrive/
├── dataset/                              ← уже есть
│   ├── airy/, dark/, dramatic/,
│   ├── golden_hour/, minimalist/,
│   ├── monochrome/, neon/, vintage/
└── efficientnet_b0_styles.pth            ← загрузить из ml-service/weights/
```

Артефакты экспериментов будут сохраняться в `MyDrive/diploma_out/` — папка создастся автоматически.

## Шаг 0. Setup — выполнить один раз в начале сессии

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Клонируем ветку с экспериментами
!git clone -b claude/sleepy-gould-93f13f https://github.com/tvrvs91/visual-style-classifier.git
%cd visual-style-classifier

In [ ]:
# Доп. зависимости (поверх того что Colab уже даёт)
!pip install -q -r training/requirements-experiments.txt

In [ ]:
# Удобные пути — потом используются во всех экспериментах
import os
DRIVE = '/content/drive/MyDrive'
WEIGHTS = f'{DRIVE}/efficientnet_b0_styles.pth'
DATA = '/content/dataset'                       # сюда симлинки положим
OUT = f'{DRIVE}/diploma_out'                    # артефакты в Drive
os.makedirs(OUT, exist_ok=True)

assert os.path.exists(WEIGHTS), f'нет файла весов: {WEIGHTS} — загрузи в Drive'
assert os.path.exists(f'{DRIVE}/dataset'), f'нет датасета: {DRIVE}/dataset'
print('Setup OK')
print(f'  weights : {WEIGHTS}')
print(f'  dataset : {DRIVE}/dataset')
print(f'  out     : {OUT}')

In [ ]:
# Разделение датасета на train/val (через симлинки, Drive не меняется)
!python training/make_split.py \
    --src {DRIVE}/dataset \
    --dst {DATA} \
    --val-ratio 0.15 --seed 42

## E3. Score-card per-class анализ

Самый быстрый, GPU не нужен. Прогоняем первым — заодно проверит что split правильный и зависимости в порядке.

**Что получаем:**
- `radar.png` — 8 радарных чартов, по одному на класс
- `violins.png` — распределения 5 score-card метрик по классам
- `stats.csv` — mean/std/median (для таблиц в дипломе)
- `anova.json` — статистическая значимость каждой метрики
- `baseline.json` — accuracy logreg/RF на 5 фичах (нижняя планка качества)

In [ ]:
!python training/score_card_class.py \
    --data-dir {DATA} \
    --out-dir {OUT}/scorecard \
    --splits train val

In [ ]:
# Беглый просмотр результатов
from IPython.display import Image, display
import json
display(Image(f'{OUT}/scorecard/radar.png'))
display(Image(f'{OUT}/scorecard/violins.png'))
print(json.dumps(json.load(open(f'{OUT}/scorecard/baseline.json')), indent=2, ensure_ascii=False)[:1500])

## E1. Grad-CAM визуализация attention

Грузит обученные веса, считает heatmap'ы Grad-CAM на последнем свёрточном блоке EfficientNet-B0. По 6 изображений на класс + топ-10 уверенно неправильных предсказаний.

**Что получаем:**
- `gradcam_grid.png` — 8×6 общий grid
- `gradcam_<class>.png` — отдельные коллажи по классам
- `gradcam_errors.png` — топ-10 ошибок с heatmap'ом куда смотрела модель
- `notes.json` — путь/класс/confidence по каждому изображению

In [ ]:
!python training/gradcam_viz.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/gradcam \
    --per-class 6 --errors 10

In [ ]:
display(Image(f'{OUT}/gradcam/gradcam_grid.png'))
display(Image(f'{OUT}/gradcam/gradcam_errors.png'))

## E2. Embeddings + t-SNE / UMAP

Извлекает 1280-мерные эмбеддинги (выход global avgpool, до классифицирующей головы), считает разделимость кластеров классов.

**Что получаем:**
- `tsne.png`, `umap.png` — 2D-проекции
- `metrics.json` — silhouette, Davies-Bouldin, kNN5-accuracy
- `centroid_similarity.png/.csv` — матрица 8×8 cosine similarity между центроидами
- `embeddings.npy`, `labels.npy` — сырые данные на случай если захотим что-то ещё посчитать

In [ ]:
!python training/embed_viz.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/embed \
    --split val --umap

In [ ]:
display(Image(f'{OUT}/embed/tsne.png'))
display(Image(f'{OUT}/embed/umap.png'))
display(Image(f'{OUT}/embed/centroid_similarity.png'))
print(json.dumps(json.load(open(f'{OUT}/embed/metrics.json')), indent=2, ensure_ascii=False))

## E4. Калибровка вероятностей (temperature scaling)

Подбирает скаляр T так, чтобы softmax(logits/T) был лучше калиброван (ECE ниже). Метод Guo et al. 2017.

**Что получаем:**
- `reliability_combined.png` — reliability diagram до/после рядом
- `metrics.json` — ECE/MCE/Brier/NLL до и после
- `temperature.txt` — найденное T (одно число; если выигрыш значим — вкатим в ml-service)

In [ ]:
!python training/calibration.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/calibration \
    --vector-scaling

In [ ]:
display(Image(f'{OUT}/calibration/reliability_combined.png'))
print('Найденное T:', open(f'{OUT}/calibration/temperature.txt').read())
print(json.dumps(json.load(open(f'{OUT}/calibration/metrics.json')), indent=2, ensure_ascii=False))

## E5. CLIP linear-probe + zero-shot

Сравнение с foundation-моделью CLIP ViT-B/32 (OpenAI). Linear probe = sklearn logreg поверх замороженного CLIP. Zero-shot = классификация через текстовые prompts «a {style} photo» без обучения.

Качает ~350 МБ модели CLIP. **Включить GPU для скорости** (на CPU будет ~20 мин).

**Что получаем:**
- `comparison_table.csv` — сводка: EffNet-B0 vs CLIP-LP vs CLIP-ZS
- `confusion_matrix_lp.png`, `confusion_matrix_zs.png`
- `tsne_clip.png` — CLIP embeddings в 2D (можно сравнить с E2)

In [ ]:
!python training/clip_probe.py \
    --data-dir {DATA} \
    --out-dir {OUT}/clip \
    --model ViT-B-32 --pretrained openai

In [ ]:
import csv
with open(f'{OUT}/clip/comparison_table.csv') as f:
    for row in csv.reader(f):
        print(row)
display(Image(f'{OUT}/clip/confusion_matrix_lp.png'))
display(Image(f'{OUT}/clip/tsne_clip.png'))

## E6. Multi-label fine-tune (опциональный, требует GPU)

Переобучает голову + всю модель на multi-label метках. Метки автоматически расширяются через score-card правила (например, фото с saturation < 0.10 получает дополнительную метку `monochrome` поверх gold-класса).

Тёплый старт от v3-весов → быстро сходится. ~30 мин на T4.

**Что получаем:**
- `efficientnet_b0_multilabel.pth` — новые веса
- `thresholds.json` — per-class threshold
- `metrics.json` — mAP, per-class F1, exact-match, Hamming
- `co_label_stats.json` — какие комбинации меток получились

In [ ]:
# Сначала проверим, что GPU видно
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
!python training/train_multilabel.py \
    --data-dir {DATA} \
    --out-dir {OUT}/multilabel \
    --init-weights {WEIGHTS} \
    --epochs 25 --head-epochs 5 \
    --batch-size 32 --num-workers 2 \
    --patience 7

In [ ]:
metrics = json.load(open(f'{OUT}/multilabel/metrics.json'))
print('mAP:', metrics['mAP'])
print('macro F1:', metrics['macro_f1'])
print('exact_match:', metrics['exact_match_accuracy'])
print('hamming:', metrics['hamming_accuracy'])
print()
print('Per-class:')
for cls, m in metrics['per_class'].items():
    print(f"  {cls:14s}  F1={m['f1']:.3f}  AP={m['AP']:.3f}  thr={m['threshold']:.2f}")

## Финальная сводка

После прохода всех ячеек в `MyDrive/diploma_out/` будет такая структура:

```
diploma_out/
├── scorecard/   (E3)
├── gradcam/     (E1)
├── embed/       (E2)
├── calibration/ (E4)
├── clip/        (E5)
└── multilabel/  (E6)
```

Дальше пришли мне путь к этой папке (или просто закоммить картинки + JSON в репо в `docs/figures/` и `docs/figures/json/`) — я переведу результаты в текст для дипломного отчёта.

**Минимум, который нужен для каждого эксперимента в отчёте:**

| Эксперимент | Картинки в `docs/figures/` | JSON в `docs/figures/json/` |
|---|---|---|
| E3 | `radar.png`, `violins.png` | `stats.csv`, `anova.json`, `baseline.json` |
| E1 | `gradcam_grid.png`, `gradcam_errors.png` | `notes.json` |
| E2 | `tsne.png`, `umap.png`, `centroid_similarity.png` | `metrics.json`, `centroid_similarity.csv` |
| E4 | `reliability_combined.png` | `metrics.json`, `temperature.txt` |
| E5 | `confusion_matrix_lp.png`, `tsne_clip.png` | `linear_probe_metrics.json`, `zero_shot_metrics.json`, `comparison_table.csv` |
| E6 | — (числа в JSON важнее) | `metrics.json`, `co_label_stats.json` |